In [1]:
%load_ext autoreload
%autoreload 2


import os
import logging
import pandas as pd
import numpy as np
import time
import torch
import wandb
import copy
from tqdm import tqdm


from src.constants import DEVICE
from src.utils.loss_func.get_loss_function import get_loss_function
from src.evaluation.mainMetricHandler import mainMetricHandler
from src.dataset.load_dataset import load_dataset, generate_K_fold_cross_validation_splits
from src.models.tools.get_classification_model import get_classification_model
from src.dataset.get_dataloader import make_dataloader   
from src.dataset.get_transforms import get_transforms
from src.utils.loss_func.get_loss_function import get_loss_function
from src.evaluation.mainMetricHandler import mainMetricHandler



C:\Users\S.P.M. de Vette\AppData\Local\Temp\ipykernel_41892\133410121.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was too old on your system - pyarrow 10.0.1 is the current minimum supported version as of this release.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
import torch

torch.cuda.is_available()


True

In [3]:
from src.config_presets.tools.get_config import get_config

config = get_config('Multi_tox')




src\config_presets\Base_config.yaml
src\config_presets\Multi_tox.yaml


In [4]:
# load the data
df_train_val, df_test = load_dataset(config)
    
dataset_split_dict = generate_K_fold_cross_validation_splits(config, df_train_val)

# cap the number of iterations, if it is less than the number of k-splits to make
n_iterations = config['data']['kFolds']['n_iterations']

# get the data transforms
train_transforms, val_transforms = get_transforms(config)
# get the loss function
loss_function = get_loss_function(config)

metricHandler = mainMetricHandler(config)



Removed patients (no image data) = 0
Train/Val dataset 872 (80.0%), Test dataset 218 (20.0%)


c:\Users\S.P.M. de Vette\.conda\envs\HNC_310\lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [5]:
train_data, val_data = dataset_split_dict[0]['train'], dataset_split_dict[0]['val']

train_loader, metadata = make_dataloader(config, train_data, train_transforms, validation_mode=False)
val_loader, _ = make_dataloader(config, val_data, val_transforms, validation_mode=True)



In [6]:
from src.models.tools.get_classification_model import get_classification_model



config['general']['resultsCurrentDirectory'] = os.getcwd()


config['model']['TransRP']['clinical_features_method'] = 'cls'

config['model']['TransRP']['vit_dim'] = 128

config['model']['linear_units'] = []

model = get_classification_model(config, metadata=metadata, save_summary=False)



c:\Users\S.P.M. de Vette\.conda\envs\HNC_310\lib\site-packages\torch\nn\modules\lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [7]:
model

MultiTox_Classifier(
  (encoder): DenseNet(
    (features): Sequential(
      (conv0): Conv3d(3, 64, kernel_size=(5, 5, 5), stride=(2, 2, 2), padding=(2, 2, 2), bias=False)
      (norm0): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu0): LeakyReLU(negative_slope=0)
      (pool0): MaxPool3d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (denseblock_1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (layers): Sequential(
            (norm1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu1): LeakyReLU(negative_slope=0)
            (conv1): Conv3d(64, 128, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
            (norm2): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu2): LeakyReLU(negative_slope=0)
            (conv2): Conv3d(128, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=Fal